In [6]:
import os, glob
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm

In [7]:

levels = ['8m', '16m', '47m', '80m', '140m', '200m', '280m']
#data_path = r"D:\!Work\IFA\!data\Urban_fluxes_NE\raw\Beijing\2024.6\Beijing tower"
data_path = r"D:\!Work\IFA\!data\Urban_fluxes_NE\raw\Beijing\*\Beijing tower"


df4levels = {}

for level in levels:
    files = glob.glob(os.path.join(data_path + '\\' + level, "*.txt"))
    df = pd.DataFrame()
    for file in tqdm(files):
        cur_df = pd.read_csv(file, sep='\t', skiprows=range(1,2), parse_dates={'date_full':['date', 'time']}).set_index('date_full')    
        df = pd.concat((df, cur_df))

    df['Tau'] = df['air_density'] * df['u*']**2

    df4levels[level] = df

100%|██████████| 92/92 [00:00<00:00, 148.23it/s]


In [8]:
%matplotlib qt
plt.figure()
#df4levels['80m']['H'].plot()

<Figure size 1600x1200 with 0 Axes>

In [9]:


#[c for c in df4levels['8m'].columns if 'co2' in c]

#%matplotlib inline

# df4levels['80m']['qc_Tau'].plot()
# df4levels['80m']['Tau'].plot()


In [10]:
export_dir = "D:/!Work/IFA/!data/Urban_fluxes_NE/processed/Beijing" 

time_shift = pd.Timedelta(8, 'h')

os.makedirs(export_dir, exist_ok=True)

for level in df4levels.keys():
    df4level = df4levels[level]
    df2export = df4level[['H', 'LE', 'Tau', 'u*', 'air_temperature', 'RH', 'air_pressure', 'air_density', 'v_unrot', 'u_unrot', 'w_unrot', 'wind_speed', 'wind_dir', 'co2_flux']] #, 'uz', 't', 'r', 'P', 'ro', 'u', 'v', 'w', 'V', 'D', 'alfa']]
    df2export.index.set_names('datetime_utc', inplace=True)

    
    df2export = df2export.rename(columns = {'H': 'Qh',
                                            'LE': 'Qle',
                                            'Tau': 'Qtau',
                                            'u*':'Ustar', 
                                            'air_temperature': 'T_air',
                                            'RH': 'RH_air',
                                            'air_pressure': 'P_air',
                                            'air_density': 'Rho_air',
                                            'v_unrot':'Wind_V',
                                            'u_unrot':'Wind_U',
                                            'w_unrot':'Wind_W',
                                            'wind_speed':'Wind_vel',
                                            'wind_dir':'Wind_dir',
                                            'co2_flux':'Qco2',
                                            })
    df2export.index = df2export.index - time_shift
    df2export.to_csv(export_dir + '/' + 'Beijing' + '_' + level +  '.csv')

    